# 03 - Assign Cluster Labels to ALL Trajectories  (Stage 3)

**Purpose.** Load the chosen k-means model **and the saved StandardScaler**, and
assign a `cluster_label` to every trajectory by nearest-centroid lookup
(`predict`, no iteration). The scaler from notebook 02 must be applied to the raw
features before `predict`, so every trajectory is standardized with the *same*
means/stds the model was trained on.

- `complete` and `partial` trajectories are labelled (for `partial`, the last
  known position was already used as the proxy for the missing later day(s) in
  Stage 1).
- `early_loss` trajectories have no usable features -> `cluster_label = -1`.

If `config.GROUP_MAP` is set, a merged `cluster_group` column is also added;
otherwise `cluster_group` mirrors `cluster_label`.

**Input.** `data/features.parquet`, `data/kmeans_models/kmeans_k{BEST_K}.pkl`,
`data/kmeans_models/scaler.pkl`.
**Output.** `data/labeled_trajectories.parquet`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print("sampling days:", C.DAYS, "| feature space:", C.FEATURE_SPACE)

project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w
sampling days: [30, 50, 100, 150, 180] | feature space: zscore


## 3.1  Choose k and load the model + scaler

In [2]:
BEST_K = 70          # the manual fit from notebook 1.5 (papermill: -p BEST_K <k>)


In [3]:
import joblib
# Load the model saved by notebook 1.5, NOT the ones in MODELS_DIR itself:
# those are written by notebook 02 and were fit on different data / weights.
MODEL_DIR = C.MODELS_DIR / f"manual_1.5_k{BEST_K}"
km     = joblib.load(MODEL_DIR / f"kmeans_k{BEST_K}.pkl")
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
print("loaded", MODEL_DIR.name, "with", km.n_clusters, "clusters and its own scaler")

# the weighting must match the one the model was fit with
_W_fit = np.load(MODEL_DIR / "weights_W.npy")
assert np.allclose(_W_fit, P.feature_weight_vector()), (
    "config.DAY_WEIGHTS does not match the weights this model was fit with")
print("config.DAY_WEIGHTS matches the fit")


loaded manual_1.5_k70 with 70 clusters and its own scaler
config.DAY_WEIGHTS matches the fit


## 3.2  Predict labels for every labellable trajectory

In [4]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status != "early_loss"
# Standardize with the SAME scaler, then apply the SAME per-day weighting the
# model was fit with (config.DAY_WEIGHTS via pipeline.feature_weight_vector()).
# The k-means centroids live in this weighted space, so predict MUST see it too;
# omitting * W would assign labels in a different space than the model was fit in.
W = P.feature_weight_vector()
X_all = scaler.transform(P.build_feature_matrix(features[labelable])) * W
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable.to_numpy()] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

cluster_label
0      113063
1      757737
2       90199
3       61078
4      175166
5      738044
6      126097
7      355704
8       57947
9      187471
10     178046
11      70916
12      49704
13      97842
14      88119
15      83999
16     910337
17     218387
18     137617
19     160581
20     121176
21     125563
22      79444
23     106049
24     417592
25     507704
26      49106
27     156983
28      91177
29     108156
30      76892
31     139686
32      47282
33      57115
34     108308
35     197686
36     166659
37     166535
38     172681
39     581019
40      60742
41     267715
42     126179
43      52916
44     108626
45      50594
46     100468
47     134078
48     153207
49      93447
50     101123
51     102517
52     163731
53     140948
54      74648
55    2889558
56      49532
57      73795
58      28838
59     109544
60    1091623
61     154997
62     238335
63      79386
64     182899
65     104461
66      45725
67     131431
68      89351
69     144719


## 3.2b  Merge into pathway groups (optional)

If `config.GROUP_MAP` is non-empty, merge the raw clusters into groups; the
result is stored in `cluster_group`. With an empty map, `cluster_group` simply
equals `cluster_label`.

In [5]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .value_counts().sort_index().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

raw cluster -> group: {0: 6, 1: 9, 2: 3, 3: 5, 4: 4, 5: 10, 6: 1, 7: 8, 8: 5, 9: 4, 10: 6, 11: 0, 12: 3, 13: 1, 14: 6, 15: 6, 16: 10, 17: 12, 18: 4, 19: 4, 20: 1, 21: 4, 22: 2, 23: 13, 24: 12, 25: 12, 26: 1, 27: 5, 28: 1, 29: 4, 30: 1, 31: 5, 32: 11, 33: 5, 34: 7, 35: 4, 36: 13, 37: 7, 38: 9, 39: 9, 40: 5, 41: 10, 42: 1, 43: 11, 44: 4, 45: 11, 46: 3, 47: 8, 48: 1, 49: 0, 50: 4, 51: 1, 52: 4, 53: 4, 54: 2, 55: 10, 56: 6, 57: 1, 58: 11, 59: 7, 60: 10, 61: 4, 62: 13, 63: 13, 64: 4, 65: 12, 66: 5, 67: 1, 68: 1, 69: 3}
cluster_group
0      164363
1     1238770
2      154092
3      385090
4     1944564
5      579276
6      512759
7      384387
8      489782
9     1511437
10    5897277
11     179630
12    1248144
13     590429


## 3.3  Save labelled trajectories

In [6]:
features.to_parquet(C.LABELED_FILE, index=False)
print("saved", features.shape, "->", C.LABELED_FILE)

saved (15280000, 18) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/labeled_trajectories.parquet


## 3.4  Summary

In [7]:
lbl = features[features.cluster_label >= 0]
print(f"Labelled {len(lbl):,} trajectories into {BEST_K} clusters "
      f"({(features.cluster_label==-1).sum():,} early_loss left unlabelled).")
print(f"Saved to {C.LABELED_FILE}")

Labelled 15,280,000 trajectories into 70 clusters (0 early_loss left unlabelled).
Saved to /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/labeled_trajectories.parquet
